# Stream Processing Pipeline - NYC Yellow Taxi

Esse notebook roda o pipeline inteiro de ponta a ponta: baixa os dados reais, simula a chegada em stream, valida, agrega e grava o resultado em Parquet. Da pra rodar tanto local (Jupyter) quanto no Databricks sem alterar nada, o notebook detecta sozinho onde esta rodando.

Se o csv ainda nao estiver em `data/raw/`, o notebook baixa uma amostra da NYC TLC automaticamente (via NYC Open Data, ver celula de download mais abaixo). Precisa de internet na primeira execucao. Se preferir baixar antes e separado, tem o script `scripts/baixar_dataset.py` que faz a mesma coisa.

**Basta rodar todas as celulas em ordem (Run All).** Dependencias: ver `requirements.txt` (`pip install -r requirements.txt`). Local, e importante usar Java 8, 11 ou 17 (Spark 3.5 nao roda em Java mais novo que isso).

## Parte 1: Ingestao, validacao e output

### Setup

Garante que a versao do pyspark instalada e compativel (3.5.x, que roda em Java 8/11/17). Se a maquina tiver o pyspark 4.x instalado, ele exige Java 17+, entao aqui a gente forca a versao certa. No Databricks isso nao roda, o cluster ja vem com o Spark configurado.

In [ ]:
import os
import sys
import subprocess

IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if not IS_DATABRICKS:
    precisa_instalar = False
    try:
        import pyspark
        if int(pyspark.__version__.split(".")[0]) >= 4:
            precisa_instalar = True  # pyspark 4.x exige o Java 17+, preferimos 3.5.x
    except ImportError:
        precisa_instalar = True

    if precisa_instalar:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pyspark==3.5.1", "pandas"])

### Ambiente: local x Databricks

Detecta automaticamente se esta rodando no Databricks (variavel de ambiente `DATABRICKS_RUNTIME_VERSION`) e escolhe os caminhos certos.

No Databricks usamos o disco local do cluster (`/tmp`), em vez do DBFS (`/dbfs`, `dbfs:/...`). Varios workspaces novos (principalmente o Databricks Free Edition) vem com o DBFS root publico desabilitado por padrao, entao `/dbfs/FileStore` da erro `DBFS_DISABLED` tanto pra ler quanto pra escrever, sem depender de nenhuma config extra tipo Volume/catalogo. `/tmp` funciona sempre, sem precisar de permissao especial. A unica pegadinha: isso so funciona porque o cluster e single-node (o que e o normal pra esse tipo de workspace); num cluster com varios workers, os arquivos em `/tmp` do driver nao apareceriam pros workers.

In [ ]:
if IS_DATABRICKS:
    PYTHON_BASE = "/tmp/nyc_taxi_streaming"
    SPARK_BASE = "file:/tmp/nyc_taxi_streaming"  # "file:" forca o Spark a usar disco local em vez de DBFS
else:
    PYTHON_BASE = "."
    SPARK_BASE = "."

RAW_PATH = f"{PYTHON_BASE}/data/raw/yellow_tripdata.csv"
STREAM_SOURCE_PATH_PY = f"{PYTHON_BASE}/data/stream_source"
STREAM_SOURCE_PATH_SPARK = f"{SPARK_BASE}/data/stream_source"
OUTPUT_PATH = f"{SPARK_BASE}/output/parquet"
CHECKPOINT_PATH = f"{SPARK_BASE}/output/checkpoints"

print("rodando no Databricks:" if IS_DATABRICKS else "rodando local:", IS_DATABRICKS)
print("RAW_PATH:", RAW_PATH)
print("STREAM_SOURCE_PATH_SPARK:", STREAM_SOURCE_PATH_SPARK)
print("OUTPUT_PATH:", OUTPUT_PATH)

### Sessao Spark

No Databricks a `spark` ja vem criada, `getOrCreate()` so reaproveita ela. Se der erro tipo `UnsupportedClassVersionError` rodando local, e porque a versao do Java instalada nao bate com a do pyspark (Spark 3.5.x precisa de Java 8, 11 ou 17). Se der erro mencionando `winutils`/`HADOOP_HOME`/`NativeIO`, e o problema conhecido do Spark local no Windows precisar do `winutils.exe`; costuma ser so aviso (WARN) e nao impede o pipeline de rodar, mas se travar mesmo, mais facil rodar em WSL, Linux/Mac ou direto no Databricks (foi assim que testamos).

In [ ]:
from pyspark.sql import SparkSession

try:
    spark = SparkSession.builder.appName("nyc-taxi-streaming").getOrCreate()
except Exception as erro:
    print("Erro ao criar a sessao Spark.")
    print("UnsupportedClassVersionError => versao de Java incompativel (usar Java 8, 11 ou 17 pro Spark 3.5.x).")
    print("Mencao a winutils/HADOOP_HOME/NativeIO => problema conhecido do Spark local no Windows.")
    raise erro

### Dados de entrada: download

A TLC hoje disponibiliza os arquivos oficiais so em Parquet, entao usamos a mesma base exportada em CSV pelo NYC Open Data (https://data.cityofnewyork.us), que e o proprio portal de dados da prefeitura de Nova York alimentado pela TLC. Baixamos uma amostra (primeira semana de janeiro/2023) direto pra `RAW_PATH`, se ainda nao existir.

In [ ]:
import urllib.parse
import urllib.request

DATASET_BASE_URL = "https://data.cityofnewyork.us/resource/4b4i-vvec.csv"  # NYC Open Data - Yellow Taxi 2023
DATASET_PARAMS = {
    "$limit": "5000",
    "$where": "tpep_pickup_datetime between '2023-01-01T00:00:00' and '2023-01-07T23:59:59'",
    "$order": "tpep_pickup_datetime",
}

def baixar_dataset(destino):
    url = DATASET_BASE_URL + "?" + urllib.parse.urlencode(DATASET_PARAMS)
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    print(f"baixando dataset de {url}")
    urllib.request.urlretrieve(url, destino)
    print(f"salvo em {destino}")

if not os.path.exists(RAW_PATH):
    try:
        baixar_dataset(RAW_PATH)
    except Exception as erro:
        print("Nao foi possivel baixar o dataset.")
        print("Rode 'python scripts/baixar_dataset.py' antes, ou baixe manualmente:")
        print(DATASET_BASE_URL + "?" + urllib.parse.urlencode(DATASET_PARAMS))
        print(f"e salve em {RAW_PATH}")
        raise erro
else:
    print(f"usando o csv ja baixado em {RAW_PATH}")

### Schema dos dados

Definido na mao porque em streaming o Spark exige isso. Segue as colunas do csv atual do NYC Open Data/TLC (esquema pos-2016, com zonas `pulocationid`/`dolocationid` em vez de latitude/longitude).

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

schema = StructType([
    StructField("vendorid", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("ratecodeid", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("pulocationid", IntegerType(), True),
    StructField("dolocationid", IntegerType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True),
])

### Simulando a chegada dos dados

A TLC libera os dados em lote (arquivo fechado), entao pra simular um stream a gente parte o csv em pedacos menores e vai escrevendo eles aos poucos na pasta de origem. O Spark fica de olho nessa pasta com o `readStream`. Os valores padrao aqui (`intervalo_segundos=2`, `limite_arquivos=10`) sao pensados pra demo rapida, ajustar se quiser um teste maior.

In [ ]:
import time
import pandas as pd

def simular_stream(raw_path, destino, linhas_por_arquivo=200, intervalo_segundos=2, limite_arquivos=10):
    os.makedirs(destino, exist_ok=True)
    leitor = pd.read_csv(raw_path, chunksize=linhas_por_arquivo)
    for i, chunk in enumerate(leitor):
        if limite_arquivos and i >= limite_arquivos:
            break
        arquivo = os.path.join(destino, f"corridas_{i:05d}.csv")
        chunk.to_csv(arquivo, index=False)
        print(f"gerado {arquivo}")
        time.sleep(intervalo_segundos)

### Ingestao (readStream) e validacao (bonus 1)

Descarta corrida sem passageiro, sem distancia, com tarifa negativa/zerada, e linha com algum campo fundamental nulo (o dataset tem alguns desses casos, e um bom teste real do filtro). A pasta de origem precisa existir antes do `readStream.load()`, por isso o `makedirs` logo no comeco.

In [ ]:
from pyspark.sql import functions as F

os.makedirs(STREAM_SOURCE_PATH_PY, exist_ok=True)  # o readStream exige que a pasta ja exista

df_bruto = (
    spark.readStream
    .format("csv")
    .option("header", "true")
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss.SSS")
    .schema(schema)
    .option("maxFilesPerTrigger", 1)
    .load(STREAM_SOURCE_PATH_SPARK)
)

df_validado = (
    df_bruto
    .filter(F.col("passenger_count") > 0)
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") > 0)
    .dropna(subset=["vendorid", "tpep_pickup_datetime", "tpep_dropoff_datetime"])
)

### Output em Parquet

Aqui o `df_validado` e o ponto de entrada pra Parte 2 (agregacao com window). Por enquanto grava direto o resultado validado.

In [ ]:
query = (
    df_validado.writeStream
    .format("parquet")
    .option("path", OUTPUT_PATH)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .outputMode("append")
    .start()
)

print("query rodando, status:", query.status)

### Rodando a simulacao

A query acima ja esta rodando em background (`.start()` nao bloqueia). Agora sim disparamos a geracao dos arquivos, que vao sendo consumidos pelo stream enquanto essa celula roda.

In [ ]:
simular_stream(RAW_PATH, STREAM_SOURCE_PATH_PY, linhas_por_arquivo=200, intervalo_segundos=2, limite_arquivos=10)

# da uma folga pro Spark processar o ultimo arquivo antes de checar o resultado
time.sleep(5)
query.stop()
print("query finalizada")

### Resultado esperado

Lendo de volta o parquet gravado, pra confirmar que o pipeline processou os dados.

In [ ]:
resultado = spark.read.parquet(OUTPUT_PATH)
print("total de linhas gravadas em parquet:", resultado.count())
resultado.show(10)